# CNN Baseline

This notebook uses the shared preprocessing notebook before defining the model.

In [6]:
from pathlib import Path
from IPython import get_ipython
import copy
import json
import os
import time
from dataclasses import asdict, dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

preprocessing_notebook = Path("shared/preprocessing.ipynb")
if not preprocessing_notebook.exists():
    preprocessing_notebook = Path("../shared/preprocessing.ipynb")
get_ipython().run_line_magic("run", str(preprocessing_notebook))

evaluation_notebook = Path("shared/evaluate.ipynb")
if not evaluation_notebook.exists():
    evaluation_notebook = Path("../shared/evaluate.ipynb")
get_ipython().run_line_magic("run", str(evaluation_notebook))

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset directory: {DEFAULT_DATA_DIR}")

Using device: cpu
Project root: d:\Projects\Wildfire-Spread-Prediction-from-Satellite-Imagery
Dataset directory: d:\Projects\Wildfire-Spread-Prediction-from-Satellite-Imagery\data


# Model Architecture

In [7]:
class ConvBlock(nn.Module):
    """Conv -> BatchNorm -> ReLU, repeated twice."""

    def __init__(self, in_ch: int, out_ch: int, dropout: float = 0.0):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

In [8]:
class CNNBaseline(nn.Module):
    """Fully-convolutional encoder-decoder with NO skip connections.

    Channel progression: 12 -> 32 -> 64 -> 128 -> 256 -> 512 (bottleneck)
    -> 256 -> 128 -> 64 -> 32 -> 1

    This matches the team's U-Net in depth/width so that comparing
    CNNBaseline vs. UNet isolates the effect of skip connections alone.
    """

    def __init__(self, in_channels: int = 12, base_channels: int = 32, dropout: float = 0.2):
        super().__init__()
        c = base_channels  # 32

        # Encoder
        self.enc1 = ConvBlock(in_channels, c)          # 64x64 -> 64x64,  12->32
        self.enc2 = ConvBlock(c, c * 2)                # 32x32,           32->64
        self.enc3 = ConvBlock(c * 2, c * 4)             # 16x16,           64->128
        self.enc4 = ConvBlock(c * 4, c * 8)             # 8x8,             128->256
        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = ConvBlock(c * 8, c * 16, dropout=dropout)  # 4x4, 256->512

        # Decoder (NO concatenation with encoder features — the key
        # architectural difference vs. U-Net)
        self.up4 = nn.ConvTranspose2d(c * 16, c * 8, kernel_size=2, stride=2)
        self.dec4 = ConvBlock(c * 8, c * 8, dropout=dropout)

        self.up3 = nn.ConvTranspose2d(c * 8, c * 4, kernel_size=2, stride=2)
        self.dec3 = ConvBlock(c * 4, c * 4, dropout=dropout)

        self.up2 = nn.ConvTranspose2d(c * 4, c * 2, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(c * 2, c * 2)

        self.up1 = nn.ConvTranspose2d(c * 2, c, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(c, c)

        self.out_conv = nn.Conv2d(c, 1, kernel_size=1)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        e1 = self.enc1(x)                 # (B, 32, 64, 64)
        e2 = self.enc2(self.pool(e1))      # (B, 64, 32, 32)
        e3 = self.enc3(self.pool(e2))      # (B, 128, 16, 16)
        e4 = self.enc4(self.pool(e3))      # (B, 256, 8, 8)

        b = self.bottleneck(self.pool(e4))  # (B, 512, 4, 4)

        d4 = self.dec4(self.up4(b))         # (B, 256, 8, 8)  -- no concat with e4
        d3 = self.dec3(self.up3(d4))        # (B, 128, 16, 16) -- no concat with e3
        d2 = self.dec2(self.up2(d3))        # (B, 64, 32, 32)  -- no concat with e2
        d1 = self.dec1(self.up1(d2))        # (B, 32, 64, 64)  -- no concat with e1

        return self.out_conv(d1)            # (B, 1, 64, 64) raw logits


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Quick sanity check on architecture shape
_test_model = CNNBaseline(in_channels=NUM_CHANNELS)
_test_out = _test_model(torch.randn(2, NUM_CHANNELS, 64, 64))
assert _test_out.shape == (2, 1, 64, 64), f"Unexpected output shape: {_test_out.shape}"
print(f"Architecture OK. Output shape: {tuple(_test_out.shape)}")
print(f"Total trainable parameters: {count_params(_test_model):,}")
del _test_model, _test_out

Architecture OK. Output shape: (2, 1, 64, 64)
Total trainable parameters: 6,982,273


# Loss function — masked, weighted BCE + Dice

In [9]:
def masked_weighted_bce(logits: torch.Tensor, targets: torch.Tensor, mask: torch.Tensor, pos_weight: torch.Tensor) -> torch.Tensor:
    """BCE-with-logits, computed only over pixels where mask == 1.

    targets may contain -1 for uncertain pixels; these are excluded via
    `mask` so their exact numeric value doesn't matter, but we clamp to
    [0, 1] defensively to keep the loss numerically well-behaved.
    """
    targets_clamped = targets.clamp(0, 1)
    per_pixel = F.binary_cross_entropy_with_logits(
        logits, targets_clamped, pos_weight=pos_weight, reduction="none"
    )
    per_pixel = per_pixel * mask
    return per_pixel.sum() / mask.sum().clamp(min=1.0)

def masked_dice_loss(logits: torch.Tensor, targets: torch.Tensor, mask: torch.Tensor, smooth: float = 1.0) -> torch.Tensor:
    """Soft Dice loss, computed only over valid (masked) pixels."""
    probs = torch.sigmoid(logits) * mask
    targets_clamped = targets.clamp(0, 1) * mask

    intersection = (probs * targets_clamped).sum(dim=(1, 2, 3))
    union = probs.sum(dim=(1, 2, 3)) + targets_clamped.sum(dim=(1, 2, 3))
    dice = (2.0 * intersection + smooth) / (union + smooth)
    return 1.0 - dice.mean()

def combined_loss(logits, targets, mask, pos_weight, bce_weight: float = 0.5, dice_weight: float = 0.5):
    bce = masked_weighted_bce(logits, targets, mask, pos_weight)
    dice = masked_dice_loss(logits, targets, mask)
    return bce_weight * bce + dice_weight * dice, bce.item(), dice.item()


# --- Quick correctness test on the loss functions ---
_logits = torch.tensor([[[[5.0, -5.0], [5.0, -5.0]]]])   # confident correct predictions
_targets = torch.tensor([[[[1.0, 0.0], [1.0, -1.0]]]])    # last pixel uncertain
_mask = (_targets != -1).float()
_pw = torch.tensor(1.0)
_loss, _bce, _dice = combined_loss(_logits, _targets, _mask, _pw)
assert _loss.item() < 0.1, f"Expected near-zero loss for confident correct predictions, got {_loss.item()}"
print(f"Loss sanity check OK -> combined={_loss.item():.4f}, bce={_bce:.4f}, dice={_dice:.4f}")

Loss sanity check OK -> combined=0.0054, bce=0.0067, dice=0.0040


# Training loop

In [10]:
@dataclass
class TrainConfig:
    lr: float = 1e-3
    weight_decay: float = 1e-4
    batch_size: int = 32
    max_epochs: int = 60
    patience: int = 8          # early stopping patience (epochs without val improvement)
    grad_clip: float = 1.0
    bce_weight: float = 0.5
    dice_weight: float = 0.5

def train_model(config: TrainConfig, train_loader, val_loader, pos_weight: float, verbose: bool = True):
    model = CNNBaseline(in_channels=NUM_CHANNELS).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
    pw_tensor = torch.tensor(pos_weight, device=device)

    history = {"train_loss": [], "val_loss": [], "val_f1": [], "val_iou": []}
    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(config.max_epochs):
        # --- train ---
        model.train()
        train_losses = []
        for x, y, mask in train_loader:
            x, y, mask = x.to(device), y.to(device), mask.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss, _, _ = combined_loss(logits, y, mask, pw_tensor, config.bce_weight, config.dice_weight)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
            optimizer.step()
            train_losses.append(loss.item())

        # --- validate ---
        model.eval()
        val_losses = []
        all_true, all_prob, all_mask = [], [], []
        with torch.no_grad():
            for x, y, mask in val_loader:
                x, y, mask = x.to(device), y.to(device), mask.to(device)
                logits = model(x)
                loss, _, _ = combined_loss(logits, y, mask, pw_tensor, config.bce_weight, config.dice_weight)
                val_losses.append(loss.item())
                all_true.append(y.cpu().numpy())
                all_prob.append(torch.sigmoid(logits).cpu().numpy())
                all_mask.append(mask.cpu().numpy())

        val_metrics = compute_metrics(
            np.concatenate(all_true), np.concatenate(all_prob), np.concatenate(all_mask)
        )

        train_loss = float(np.mean(train_losses))
        val_loss = float(np.mean(val_losses))
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_f1"].append(val_metrics["f1"])
        history["val_iou"].append(val_metrics["iou"])

        scheduler.step(val_loss)

        if verbose:
            print(f"Epoch {epoch+1:3d}/{config.max_epochs} | "
                  f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
                  f"val_f1={val_metrics['f1']:.4f} val_iou={val_metrics['iou']:.4f} "
                  f"lr={optimizer.param_groups[0]['lr']:.2e}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= config.patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch+1} (no val improvement for {config.patience} epochs)")
                break

    model.load_state_dict(best_state)
    return model, history, best_val_loss


# Hyperparameter tuning

In [11]:
SEARCH_SPACE = [
    TrainConfig(lr=1e-3, bce_weight=0.5, dice_weight=0.5, max_epochs=15, patience=5),
    TrainConfig(lr=5e-4, bce_weight=0.5, dice_weight=0.5, max_epochs=15, patience=5),
    TrainConfig(lr=1e-3, bce_weight=0.7, dice_weight=0.3, max_epochs=15, patience=5),
    TrainConfig(lr=1e-3, bce_weight=0.3, dice_weight=0.7, max_epochs=15, patience=5),
]


def run_hyperparameter_search(search_space, train_loader, val_loader, pos_weight):
    results = []
    for index, config in enumerate(search_space, start=1):
        print(f"\n=== Search config {index}/{len(search_space)}: {asdict(config)} ===")
        _, _, best_val_loss = train_model(
            config, train_loader, val_loader, pos_weight, verbose=False
        )
        results.append({"config": asdict(config), "best_val_loss": best_val_loss})
        print(f"-> best_val_loss = {best_val_loss:.4f}")
    return results

# Optional: run this search using train_loader and val_loader only.
# search_results = run_hyperparameter_search(
#     SEARCH_SPACE, train_loader, val_loader, pos_weight
# )
# with open(PROJECT_ROOT / "results" / "cnn_baseline_hparam_search.json", "w") as file:
#     json.dump(search_results, file, indent=2)
# best = min(search_results, key=lambda result: result["best_val_loss"])
# print("Best config:", best)

# Train the final model with the chosen configuration

In [12]:
FINAL_CONFIG = TrainConfig(
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=32,
    max_epochs=60,
    patience=8,
    grad_clip=1.0,
    bce_weight=0.5,
    dice_weight=0.5,
)

train_loader, val_loader, test_loader, pos_weight = get_dataloaders(
    data_dir=DEFAULT_DATA_DIR,
    batch_size=FINAL_CONFIG.batch_size,
    augment_train=True,
    num_workers=0,
)

start_time = time.perf_counter()
model, history, best_val_loss = train_model(
    FINAL_CONFIG, train_loader, val_loader, pos_weight
)
training_time_sec = time.perf_counter() - start_time

print(f"\nTraining complete in {training_time_sec:.1f}s. Best val loss: {best_val_loss:.4f}")

Epoch   1/60 | train_loss=0.8248 val_loss=0.9442 val_f1=0.2017 val_iou=0.1122 lr=1.00e-03


KeyboardInterrupt: 

# Training curves

In [13]:
plot_training_curves(
    {"train_loss": history["train_loss"], "val_loss": history["val_loss"]},
    model_name="CNN Baseline"
)
plot_training_curves(
    {"val_f1": history["val_f1"], "val_iou": history["val_iou"]},
    model_name="CNN Baseline (validation F1 / IoU)"
)

NameError: name 'history' is not defined

# Test-set evaluation (shared model)

In [ ]:
num_params = count_params(model)

results = evaluate_model(
    model,
    test_loader,
    device,
    model_name="cnn_baseline",
    training_time_sec=training_time_sec,
    num_params=num_params,
)

print(json.dumps(
    {key: value for key, value in results["metrics"].items() if key != "confusion_matrix"},
    indent=2,
))
print(f"Params: {num_params:,} | Training time: {training_time_sec:.1f}s")

RESULTS_PATH = PROJECT_ROOT / "results" / "cnn_baseline_metrics.json"
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
save_results(results, str(RESULTS_PATH))

plot_confusion_matrix(
    results["metrics"]["confusion_matrix"],
    model_name="CNN Baseline",
)
plot_prediction_grid(
    model,
    test_loader,
    device,
    model_name="CNN Baseline",
    n_samples=5,
)

# Save the trained model checkpoint

In [ ]:
CHECKPOINT_PATH = PROJECT_ROOT / "checkpoints" / "cnn_baseline_best.pt"
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "config": asdict(FINAL_CONFIG),
        "num_params": num_params,
        "test_metrics": results["metrics"],
    },
    CHECKPOINT_PATH,
)
print(f"Checkpoint saved to {CHECKPOINT_PATH}")